# Phase 5 ΔE Headline — n=10 Graduation Experiment (Colab)

**Active phase:** 5
**Headline metric per [phase-5-unified-design.md:256-281](https://github.com/Dypatterson/Neuro-AI/blob/main/notes/emergent-codebook/phase-5-unified-design.md):** `ΔE = E_content-prior(q*) − E_role-prior(q*)`, mean across n_seeds × n_cues with 95% bootstrap CI.
**Required controls per [phase-5-unified-design.md:285-293](https://github.com/Dypatterson/Neuro-AI/blob/main/notes/emergent-codebook/phase-5-unified-design.md):** random-schema (random_K1), K=1 (run as primary), no-prior γ=0 (role_K1_g0 vs content_K1_g0). No-schema-store control is "should-do" only and is not implemented.
**Last verified result:** Decision-5 spike, n=1, ΔE = +2.6e-5 at K=1, N=12 atoms (post-death substrate, NOT the A+B+A1' substrate).
**Why this experiment now:** the actual Phase 5 graduation headline has never been run at n=10 multi-seed, and never on the A+B+A1' substrate. The four-agent sanity check on 2026-05-21 surfaced this — see commit [1a577ac](https://github.com/Dypatterson/Neuro-AI/commit/1a577ac).

## Pre-committed gate (binding)

Per [notes/notes/2026-05-21-phase5-headline-magnitude-floor.md](https://github.com/Dypatterson/Neuro-AI/blob/main/notes/notes/2026-05-21-phase5-headline-magnitude-floor.md):

> **`mean ΔE ≥ 5.5e-3` AND `bootstrap 95% CI lower bound > 0`**

The magnitude floor (5.5e-3) is derived ex ante from the substrate's energy noise scale at D=4096, β=10, N=1064. Decision-5's +2.6e-5 was below the noise scale at N=12; this floor tests the design spec's prediction that "larger N_schemas should yield larger magnitude."

## Pre-committed outcome table

| mean ΔE | CI lower | Verdict |
|---|---|---|
| ≥ 5.5e-3 | > 0 | **Phase 5 GRADUATES** |
| (0, 5.5e-3) | > 0 | **Graduation-unattained — directional but sub-noise** (the specific failure mode the floor guards against) |
| ≥ 5.5e-3 | crosses 0 | Graduation-unattained — meaningful magnitude but unreliable across seeds |
| < 0 with CI < 0 | — | Architectural assumption falsified (role-prior worse than content-prior) |

## Pipeline

| Cell | Action | Wall time |
|---|---|---|
| 1–4 | Setup + sanity | ~2 min |
| 5 | Build 10 A+B+A1' substrate snapshots in parallel (each run is a full Phase 3+4 cycle with `--snapshot-steps 1800`) | ~30 min |
| 6 | Run headline experiment against each snapshot in parallel (4 controls per seed) | ~10 min |
| 7 | Aggregator: mean ΔE + bootstrap 95% CI + magnitude-floor check + controls verification | <1 min |
| 8 | Copy to Drive | <1 min |

In [ ]:
# 1. Clone the repo.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

import subprocess
markers = [
    ('5.5e-3',                    'notes/notes/2026-05-21-phase5-headline-magnitude-floor.md', 'magnitude-floor pre-commit'),
    ('mean_delta_e_content_minus_role',           'experiments/40_phase5_branching.py',                       'headline-mode aggregator'),
    ('--metastability-obs-rate',                  'experiments/19_phase34_integrated.py',                     'exp 19 CLI (substrate construction)'),
    ('def save_substrate_snapshot',               'src/energy_memory/phase4/snapshot.py',                     'snapshot save primitive'),
]
for marker, fpath, label in markers:
    r = subprocess.run(['grep', '-n', '-e', marker, fpath], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'MISSING'
    print(f'{status:8s} {label}: {marker}')
    if r.stdout:
        print(f'  {r.stdout.strip().splitlines()[0]}')

In [ ]:
# 2. Mount Drive + stage the phase3c codebook.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q datasets

In [ ]:
# 4. CPU-only sanity (parent must NOT touch CUDA).
import sys, os, gc
sys.path.insert(0, 'src')

from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent CPU load):', cb.shape, cb.dtype, cb.device)
del cb; gc.collect()

# Verify exp 40 headline mode exists.
import subprocess
r = subprocess.run(['python', 'experiments/40_phase5_branching.py', '--help'],
                   env={**os.environ, 'PYTHONPATH': '/content/Neuro-AI/src'},
                   capture_output=True, text=True, cwd='/content/Neuro-AI')
assert 'headline' in r.stdout, 'exp 40 missing headline mode'
print('exp 40 headline mode: OK')

# Pre-warm wikitext cache so subprocess workers do not race.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
del splits; gc.collect()
print('cache warmed.')

In [ ]:
# 4c. GPU info (still no CUDA init in parent).
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before launching workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

## Step A — Build A+B+A1' substrate snapshots for 10 seeds (~30 min)

In [ ]:
# 5. Launch 10 substrate-construction workers in parallel.
# Each saves a snapshot at step 1800 under reports/phase5_headline_substrate_seed{N}/snapshots/

SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
N_CUES = 3000
RUN_TAG = 'phase5_headline_substrate'

# A+B+A1' pre-committed substrate parameters.
ALPHA_ANTI = 1.0
COVERAGE_LAMBDA = 1.0
COVERAGE_EMA_RATE = 0.01
REPULSION_STEP_SIZE = 100.0

import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path(f'reports/{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed):
    out_dir = f'reports/{RUN_TAG}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'experiments/19_phase34_integrated.py',
        '--device', 'cuda',
        '--updater-kind', 'hebbian',
        '--seed', str(seed),
        '--n-cues', str(N_CUES),
        '--store-threshold', '0.3',
        '--alpha-anti', str(ALPHA_ANTI),
        '--coverage-lambda', str(COVERAGE_LAMBDA),
        '--coverage-ema-rate', str(COVERAGE_EMA_RATE),
        '--repulsion-step-size', str(REPULSION_STEP_SIZE),
        '--snapshot-steps', '1800',
        '--snapshot-scales', '4',          # W=4 is what Decision-5 was tested on
        '--output-dir', out_dir,
    ]
    print(f'launching substrate seed={seed} -> {out_dir}')
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy()), logf

procs = [launch(s) for s in SEEDS]
t0 = time.time()
remaining = list(range(len(procs)))
while remaining:
    still = []
    for i in remaining:
        if procs[i][0].poll() is None:
            still.append(i)
        else:
            procs[i][1].close()
            print(f'  seed {SEEDS[i]} done at {(time.time()-t0)/60:.1f} min')
    remaining = still
    if remaining:
        time.sleep(60)
print(f'SUBSTRATE BUILD DONE in {(time.time()-t0)/60:.1f} min')

# Quick check: confirm each seed produced a W=4 snapshot at step 1800.
missing = []
for s in SEEDS:
    snap = f'reports/{RUN_TAG}_seed{s}/snapshots/phase3_phase4_w4_step1800.pt'
    if not os.path.exists(snap):
        missing.append(s)
print(f'\nSnapshots present: {len(SEEDS) - len(missing)}/{len(SEEDS)}')
if missing:
    print(f'MISSING snapshots for seeds: {missing}')

In [ ]:
# 5b. EMERGENCY kill. Interrupt the launch cell first, then run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  pid err: {e}')
print(f'killed {killed} workers')

## Step B — Run headline experiment against each snapshot (~10 min)

In [ ]:
# 6. Launch 10 headline-experiment workers in parallel.
# Each runs experiments/40_phase5_branching.py --mode headline --k-main 1
# against the corresponding seed's substrate snapshot. K=1 because Decision-5
# (line 513-517 of phase-5-unified-design.md) shows K=1 cleanly differentiates
# role-vs-content; K=4 dilutes through the bundle.

SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
N_CUES = 200    # Substantially above Decision-5's 50 for tight bootstrap CI.
RUN_TAG_SUB = 'phase5_headline_substrate'
RUN_TAG_HL  = 'phase5_headline_n10'

# Headline-experiment pre-committed parameters (per the magnitude-floor note).
K_MAIN = 1
GAMMA = 0.5
FORMULATION = 'per_pattern'   # Decision-5 line 533: production code uses per_pattern.
BETA = 10.0

import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path(f'reports/{RUN_TAG_HL}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed):
    snap = f'reports/{RUN_TAG_SUB}_seed{seed}/snapshots/phase3_phase4_w4_step1800.pt'
    out_dir = f'reports/{RUN_TAG_HL}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'experiments/40_phase5_branching.py',
        '--mode', 'headline',
        '--device', 'cuda',
        '--seed', str(seed),
        '--substrate-snapshot', snap,
        '--n-cues', str(N_CUES),
        '--k-main', str(K_MAIN),
        '--gamma', str(GAMMA),
        '--formulation', FORMULATION,
        '--beta', str(BETA),
        '--output-dir', out_dir,
    ]
    print(f'launching headline seed={seed} -> {out_dir}')
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy()), logf

procs = [launch(s) for s in SEEDS]
t0 = time.time()
remaining = list(range(len(procs)))
while remaining:
    still = []
    for i in remaining:
        if procs[i][0].poll() is None:
            still.append(i)
        else:
            procs[i][1].close()
            print(f'  seed {SEEDS[i]} done at {(time.time()-t0)/60:.1f} min')
    remaining = still
    if remaining:
        time.sleep(30)
print(f'HEADLINE EXPERIMENT DONE in {(time.time()-t0)/60:.1f} min')

In [ ]:
# 7. Aggregate the headline + check pre-committed gate.
#
# Each per-seed JSON has:
#   payload['headline_deltas']['K1'] = {
#       'n_pairs': ...,
#       'mean_delta_e_content_minus_role': ...,
#       'fraction_positive': ...,
#       'per_cue_delta': [...]    <-- this is what we bootstrap over
#   }
#   payload['conditions'] = [list of named conditions including
#     role_K1, content_K1, random_K1, role_K1_g0, content_K1_g0, ...]
#   each condition has 'per_cue_energy_unbiased_min': [list]
import json
import numpy as np
from pathlib import Path

SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
RUN_TAG_HL = 'phase5_headline_n10'
PRE_COMMITTED_FLOOR = 5.5e-3

def load_run(seed):
    p = Path(f'reports/{RUN_TAG_HL}_seed{seed}/phase5_headline_seed{seed}.json')
    if not p.exists():
        return None
    return json.loads(p.read_text())

def condition(run, name):
    for c in (run or {}).get('conditions', []):
        if c.get('name') == name:
            return c
    return None

# --- Main headline: mean ΔE_K1 = E_content_K1 − E_role_K1 ---
all_per_cue_delta = []
per_seed_means = []
for s in SEEDS:
    run = load_run(s)
    if not run:
        print(f'seed {s}: MISSING')
        continue
    hd = (run.get('headline_deltas') or {}).get('K1', None)
    if not hd:
        print(f'seed {s}: no K1 headline_deltas')
        continue
    pcd = hd.get('per_cue_delta', [])
    if not pcd:
        continue
    per_seed_means.append(hd.get('mean_delta_e_content_minus_role'))
    all_per_cue_delta.extend(pcd)
    print(f'seed {s}: n={len(pcd)} mean Delta E = {hd["mean_delta_e_content_minus_role"]:+.5f}  '
          f'frac_positive = {hd["fraction_positive"]:.2f}')

all_per_cue_delta = np.array(all_per_cue_delta)
per_seed_means = np.array(per_seed_means)
print(f'\n=== Main headline: ΔE_K1 = E_content_K1 − E_role_K1 ===')
print(f'  n_cues total       = {len(all_per_cue_delta)}')
print(f'  n_seeds            = {len(per_seed_means)}')
print(f'  mean per-cue ΔE    = {all_per_cue_delta.mean():+.5f}')
print(f'  std  per-cue ΔE    = {all_per_cue_delta.std(ddof=1):.5f}')
print(f'  mean per-seed mean = {per_seed_means.mean():+.5f}')

# Bootstrap CI across seeds (each seed contributes one mean).
rng = np.random.default_rng(2026)
boots = np.array([
    rng.choice(per_seed_means, size=len(per_seed_means), replace=True).mean()
    for _ in range(10_000)
])
ci_lo, ci_hi = np.quantile(boots, [0.025, 0.975])
print(f'  bootstrap 95% CI (per-seed means) = [{ci_lo:+.5f}, {ci_hi:+.5f}]')

# --- Pre-committed gate ---
mean_d = per_seed_means.mean()
gate_magnitude = mean_d >= PRE_COMMITTED_FLOOR
gate_ci_lower  = ci_lo > 0.0
print(f'\n=== Pre-committed gate (binding) ===')
print(f'  mean ΔE >= {PRE_COMMITTED_FLOOR:.4f}:        {"PASS" if gate_magnitude else "FAIL"}  ({mean_d:+.5f})')
print(f'  CI lower bound > 0:           {"PASS" if gate_ci_lower else "FAIL"}  ({ci_lo:+.5f})')
print()
if gate_magnitude and gate_ci_lower:
    print('PHASE 5 GRADUATES.')
elif gate_ci_lower and not gate_magnitude:
    print('Graduation-unattained: directional but sub-noise. The substrate produces')
    print('a statistically detectable preference for role-prior but at magnitude below')
    print('the per-cue noise floor. Per the pre-commit, this is NOT graduation.')
else:
    print('Graduation-unattained.')

In [ ]:
# 7b. Required-controls verification.
import numpy as np

def per_cue_energy(run, condition_name):
    c = condition(run, condition_name)
    return np.array((c or {}).get('per_cue_energy_unbiased_min', []))

print('=== Control 1: Random-schema (random_K1 vs content_K1 and vs role_K1) ===')
print('  Predicted: random ΔE ≈ 0 with CI containing zero.\n')
random_vs_content = []
random_vs_role = []
for s in SEEDS:
    run = load_run(s)
    if not run:
        continue
    e_content = per_cue_energy(run, 'content_K1')
    e_random  = per_cue_energy(run, 'random_K1')
    e_role    = per_cue_energy(run, 'role_K1')
    if len(e_content) == 0 or len(e_random) == 0:
        continue
    d_rc = float((e_content - e_random).mean())
    d_rr = float((e_random - e_role).mean()) if len(e_role) else None
    random_vs_content.append(d_rc)
    if d_rr is not None:
        random_vs_role.append(d_rr)
    print(f'  seed {s}: content − random = {d_rc:+.5f}   random − role = {d_rr if d_rr is None else f"{d_rr:+.5f}"}')

if random_vs_content:
    rvc = np.array(random_vs_content)
    print(f'\n  mean(content − random) across seeds: {rvc.mean():+.5f} ± {rvc.std(ddof=1):.5f}')
    print(f'  random is meaningfully WORSE than content if this is large positive.')

print('\n=== Control 2: No-prior γ=0 (role_K1_g0 vs content_K1_g0) ===')
print('  Predicted: ΔE ≈ 0 identically — the prior should not fire at all.\n')
g0_deltas = []
for s in SEEDS:
    run = load_run(s)
    if not run:
        continue
    hd_g0 = (run.get('headline_deltas') or {}).get('K1_g0', None)
    if hd_g0:
        g0_deltas.append(hd_g0.get('mean_delta_e_content_minus_role'))
        print(f'  seed {s}: mean ΔE_K1_g0 = {hd_g0["mean_delta_e_content_minus_role"]:+.6f}')

if g0_deltas:
    g0 = np.array(g0_deltas)
    print(f'\n  mean across seeds: {g0.mean():+.6f}')
    print(f'  If |this| < 1e-4, the no-prior control is clean.')

In [ ]:
# 8. Copy results to Drive.
import shutil, os
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for s in SEEDS:
    for tag in ('phase5_headline_substrate', 'phase5_headline_n10'):
        src = f'reports/{tag}_seed{s}'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst}/{tag}_seed{s}', dirs_exist_ok=True)
for tag in ('phase5_headline_substrate_colab', 'phase5_headline_n10_colab'):
    src = f'reports/{tag}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/{tag}', dirs_exist_ok=True)
print('results copied to', dst)
!ls {dst} | grep phase5_headline | head -25